# DEMO 1: (WorkflowAware) critical path prioritization

In this demo we demonstrate how `WorkflowAware` scheduler prioritizes critical paths of workloads which in turn helps us optimize total execution time.

In [1]:
import os
import platform
import subprocess
from enum import Enum

import numpy as np
import pandas as pd
import plotly.express as px
from PIL import Image
import plotly.graph_objects as go

## Step 1: Critical Path Bottleneck

For this demo, we're going to run a simple "random" DAG that is a combination of loose tasks and actual dependant workflows.
In this experiment we're going to limit our toplogy so it could only run 2 tasks at the time.

Because of this setup, it is unwise to execute the all possible parallel tasks at the begining, as you would be bottlenecked by remaining sequential tasks.

As we'll see shortly, the `WorkflowAware` scheduler will inteligently fill the hosts with the mix of sequential and parallel tasks, thus maximising the resource usage and finishing the workload faster!

In [1]:
dag_image = Image.open(
    "../input/synthetic_traces/random_dag_n10_edge_prob0.25_seed0_deadline_default/dag.png"
)
dag_image

NameError: name 'Image' is not defined

## Step 2: Running the Workload

In [3]:
class SchedulerEnum(Enum):
    DEFAULT_SCHEDULER = 0
    WORKFLOW_AWARE_SCHEDULER = 1

In [4]:
# DETECT JAVA HOME

os_type = platform.system()

if os_type == "Darwin":  # macOS
    os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@21"
    os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]
elif os_type == "Linux":  # Ubuntu/Linux
    java_home = os.environ.get("JAVA_HOME", "/usr/lib/jvm/java-21-openjdk-amd64")
    os.environ["JAVA_HOME"] = java_home
    os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]
else:
    print(
        f"Warning: Unsupported OS ({os_type}). JAVA_HOME may need to be set manually."
    )


In [5]:
experiment = "demo_experiments/demo_experiment_1.json"

subprocess.run(
    [
        "../OpenDCExperimentRunner/bin/OpenDCExperimentRunner",
        "--experiment-path",
        experiment,
    ]
)



 Running scenario: 0 
 Starting seed: 0 


Simulating...   0% [                                       ] 0/2 (0:00:00 / ?) 

Creating ComputeScheduler: Random with 1 hosts


 Running scenario: 1 
 Starting seed: 0 
Creating ComputeScheduler: WorkflowAware with 1 hosts


Simulating... 100% [=================================] 2/2 (0:00:00 / 0:00:00) 


CompletedProcess(args=['../OpenDCExperimentRunner/bin/OpenDCExperimentRunner', '--experiment-path', 'demo_experiments/demo_experiment_1.json'], returncode=0)

In [ ]:
fig = go.Figure()

output_path = "output/demo_experiment/raw-output/{}/seed=0/"
for scheduler_type in SchedulerEnum:
    df_task = pd.read_parquet(output_path.format(scheduler_type.value) + "task.parquet")
    df_service = pd.read_parquet(
        output_path.format(scheduler_type.value) + "service.parquet"
    )

    # calculating metrics:
    runtime = pd.to_timedelta(
        df_service.timestamp.max() - df_service.timestamp.min(), unit="ms"
    )

    print(f"Scheduler: {scheduler_type.name}")
    print(f"The workload was finished in {runtime}")

    fig.add_scatter(
        x=df_service["timestamp"] / 1000,
        y=df_service["tasks_active"],
        mode="lines",
        name=scheduler_type.name,
    )

    # print execution path
    print("Execution Path:")
    executed_tasks = (
        df_task.query("task_state == 'COMPLETED'")
        .sort_values("finish_time")["task_id"]
        .to_list()
    )

    print(executed_tasks)
    print("=" * 50 + "\n")

fig.update_layout(
    width=1000,
    height=400,
    title="Number of Active Tasks Over Time",
    xaxis_title="seconds",
    yaxis_title="active tasks",
)
fig.show()

Scheduler: DEFAULT_SCHEDULER
The workload was finished in 0 days 00:03:30
Execution Path:
[0, 1, 2, 3, 4, 7, 5, 9, 6, 8]

Scheduler: WORKFLOW_AWARE_SCHEDULER
The workload was finished in 0 days 00:02:45
Execution Path:
[3, 7, 0, 5, 1, 6, 2, 4, 8, 9]



We can see that the `WorkflowAware` scheduler was able to finish the workload faster by better utilising the available resources and prioritizing critical paths!